In [8]:
import pandas as pd
import json
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

# Connexion
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

def load_json_to_ods():
    raw_path = "raw_data"
    all_data = []

    for file in os.listdir(raw_path):
        if file.endswith(".json"):
            with open(os.path.join(raw_path, file), 'r', encoding='utf-8') as f:
                content = json.load(f)
                if isinstance(content, dict):
                    companies = content.get("results", [])
                elif isinstance(content, list):
                    companies = content
                all_data.extend(companies)

    if not all_data:
        print("Aucune donnée trouvée.")
        return

    df_ods = pd.json_normalize(all_data)
    
    # 1. Nettoyage des colonnes (points et doublons)
    df_ods.columns = [c.replace('.', '_') for c in df_ods.columns]
    df_ods = df_ods.loc[:, ~df_ods.columns.duplicated()]
    df_ods.columns = [c[:60] for c in df_ods.columns]
    df_ods = df_ods.loc[:, ~df_ods.columns.duplicated()]

    # --- ÉTAPE CRUCIALE : Convertir les dict/list en String ---
    # On parcourt chaque colonne, si elle contient des objets complexes, on les transforme en JSON string
    for col in df_ods.columns:
        if df_ods[col].apply(lambda x: isinstance(x, (list, dict))).any():
            df_ods[col] = df_ods[col].apply(lambda x: json.dumps(x) if x is not None else None)

    # 2. Insertion dans l'ODS
    try:
        df_ods.to_sql('ods_entreprises', engine, if_exists='replace', index=False)
        print(f"Succès ! ODS chargé : {len(df_ods)} lignes insérées.")
    except Exception as e:
        print(f"Erreur lors de l'insertion : {e}")

load_json_to_ods()

Succès ! ODS chargé : 20 lignes insérées.
